In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import numpy as np
import pandas as pd
import random
import math
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tqdm import tqdm

In [3]:
train_en = pd.read_csv('cleaned_english_tr.txt', sep='^^', header=None, engine='python', quoting=3)
train_hi = pd.read_csv("cleaned_hindi_tr.txt", sep='^^', header=None, engine='python', quoting=3)
val_en = pd.read_csv('cleaned_english_val.txt', sep='^^', header=None, engine='python', quoting=3)
val_hi = pd.read_csv("cleaned_hindi_val.txt", sep='^^', header=None, engine='python', quoting=3)

print(f'The training and validation files are loaded successfully:-->')
print(f'sameples train_en \n{train_en[1].head()}')
print(f'sameples train_hi \n{train_hi[1].head()}')
print('-'*100)
print(f'Data sizes train_en:{len(train_en)} train_hi:{len(train_hi)} val_en:{len(val_en)} val_hi:{len(val_hi)}')
print()

train_en_list = train_en[1].to_list()
train_hi_list = train_hi[1].to_list()
val_hi_list = val_hi[1].to_list()
val_en_list = val_en[1].to_list()

tokenizer_en = Tokenizer.from_file('tokenizer_en.json')
tokenizer_hi = Tokenizer.from_file('tokenizer_hi.json')
print(f'The tokenizer files are loaded successfully')

The training and validation files are loaded successfully:-->
sameples train_en 
0                                  But in my humanity,
1    6. Members of political coalition parties in p...
2                               This is our situation.
3    If you do business, work wisely in your financ...
4                             I've always been advised
Name: 1, dtype: object
sameples train_hi 
0                           लेकिन अपने इंसान होने में,
1    6. केन्द्र पर सत्तारूढ राजनैतिक गठबन्धन का सद्...
2                                    यह हमारी दशा है ।
3    अगर आप व्यापार करते हैं तो आर्थिक मामले में सम...
4                            मैं इस बात पर ज़ोर दूं कि
Name: 1, dtype: object
----------------------------------------------------------------------------------------------------
Data sizes train_en:380423 train_hi:380423 val_en:9755 val_hi:9755

The tokenizer files are loaded successfully


In [4]:
tokenizer_en.enable_padding(pad_id=tokenizer_en.token_to_id('[pad]'), pad_token='[pad]')
tokenizer_hi.enable_padding(pad_id=tokenizer_hi.token_to_id('[pad]'), pad_token='[pad]')

In [5]:
class customDataset(Dataset):

    def __init__(self, src, trg):
        self.src = src
        self.trg = trg

    def __len__(self):
        return len(self.src)

    def __getitem__(self, index):
        return self.src[index] , self.trg[index]


def collate_fn(batch):

    src = [item[0] for item in batch]
    trg = [item[1] for item in batch]

    src = tokenizer_en.encode_batch(src)
    trg = tokenizer_hi.encode_batch(trg)

    src = torch.LongTensor([enc.ids for enc in src])
    trg = torch.LongTensor([enc.ids for enc in trg])

    dec_input = trg[:,:-1]
    trg = trg[:,1:]

    return src, dec_input, trg
    
train_dataloader = DataLoader(
    dataset=customDataset(train_en_list, train_hi_list),
    shuffle=True,
    collate_fn=collate_fn,
    batch_size=60
)
val_dataloader = DataLoader(
    dataset=customDataset(val_en_list, val_hi_list),
    shuffle=False,
    collate_fn=collate_fn,
    batch_size=60
)

In [6]:
# Configuration:

n_heads = 4
d_model = 256
hidden_size = 512
p = 0.1
num_layers = 3

In [7]:
class PositionalEncoding(nn.Module):

    def __init__(self,d_model, p, max_len=5000):
        super().__init__()

        self.dropout = nn.Dropout(p)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0,max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position*div_term)
        pe[:, 1::2] = torch.cos(position*div_term)

        # pe : [max_length, emb_size] --> [1,max_length, emb_size]

        pe = pe.unsqueeze(0)

        self.register_buffer('pe',pe)

    def forward(self,x):

        # x : [batch, src, emb_size]
        # pe : [1, max_length, emb_size]
        return self.dropout(x + self.pe[:, 0:x.shape[1], :])

class feedForward(nn.Module):

    def __init__(self, d_model, hidden_size, p):
        super().__init__()

        self.layer1 = nn.Linear(d_model, hidden_size)
        self.layer2 = nn.Linear(hidden_size, d_model)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p)

    def forward(self, x):

        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer2(x)

        return x

In [8]:
class EncoderLayer(nn.Module):

    def __init__(self, d_model, n_heads, hidden_size, p):
        super().__init__()
        
        self.attention = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(p)

        self.feedForward = feedForward(d_model, hidden_size, p)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(p)

    def forward(self, src, src_mask):

        # src: [batch, src, emb_size]
        x1, _ = self.attention(src, src, src, key_padding_mask=src_mask)
        x = self.norm1(src + self.dropout1(x1))

        x1 = self.feedForward(x)
        x = self.norm1(x + self.dropout2(x1))

        return x

class Encoder(nn.Module):

    def __init__(self, input_size, d_model, n_heads, hidden_size, num_layers, p):
        super().__init__()

        self.d_model = d_model

        self.embeddings = nn.Embedding(input_size, d_model)
        self.pe = PositionalEncoding(d_model, p)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, hidden_size, p) for _ in range(num_layers)
        ])

    def forward(self, src, src_mask):

        # src: [batch, src]
        src = self.embeddings(src)*math.sqrt(self.d_model)
        src = self.pe(src)

        for layer in self.layers:
            src = layer(src, src_mask)

        return src # src: [batch, src, emb_size]

In [9]:
class DecoderLayer(nn.Module):

    def __init__(self, d_model, n_heads, hidden_size, p):
        super().__init__()
        
        self.selfAttention = nn.MultiheadAttention(d_model ,n_heads ,batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(p)

        self.crossAttention = nn.MultiheadAttention(d_model ,n_heads ,batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(p)

        self.feedForward = feedForward(d_model, hidden_size, p)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(p)

    def forward(self, trg, enc_out, trg_mask, src_mask):

        # src: [batch, src, emb_size]
        x1, _ = self.selfAttention(trg, trg, trg, attn_mask=trg_mask)
        x = self.norm1(trg + self.dropout1(x1))
        
        x1, _ = self.crossAttention(trg, enc_out, enc_out, key_padding_mask=src_mask)
        x = self.norm2(trg + self.dropout2(x1))

        x1 = self.feedForward(x)
        x = self.norm3(x + self.dropout3(x1))

        return x

class Decoder(nn.Module):

    def __init__(self, output_size, d_model, n_heads, hidden_size, num_layers, p):
        super().__init__()

        self.d_model = d_model

        self.embeddings = nn.Embedding(output_size, d_model)
        self.pe = PositionalEncoding(d_model, p)

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, hidden_size, p) for _ in range(num_layers)
        ])

        self.out = nn.Linear(d_model, output_size)

    def forward(self, trg, enc_out ,trg_mask,src_mask):

        # src: [batch, src]
        trg = self.embeddings(trg)*math.sqrt(self.d_model)
        trg = self.pe(trg)

        for layer in self.layers:
            trg = layer(trg, enc_out, trg_mask, src_mask)

        return self.out(trg) # trg: [batch, seq_len, output_size]

In [10]:
class Transformer(nn.Module):

    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.device = device
        self.encoder = encoder
        self.decoder = decoder
        
    def src_mask(self, src):
        return (src==tokenizer_en.token_to_id('[pad]'))
    
    def trg_mask(self, trg):
        mask = torch.triu((torch.ones((trg.shape[1], trg.shape[1]), device=self.device) * float('-inf')), diagonal=1)
        return mask
    
    def forward(self, src, dec_input):

        src_mask = self.src_mask(src)
        trg_mask = self.trg_mask(dec_input)

        # src: [batch, src]
        # trg: [batch, trg]

        enc_out = self.encoder(src, src_mask)
        output = self.decoder(dec_input, enc_out ,trg_mask, src_mask) #output: [batch, seq_len, output_size]

        return output

In [11]:
input_size = tokenizer_en.get_vocab_size()
output_size = tokenizer_hi.get_vocab_size()
print(input_size, output_size)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

encoder = Encoder(input_size, d_model, n_heads, hidden_size, num_layers, p).to(device)
decoder = Decoder(output_size, d_model, n_heads, hidden_size, num_layers, p).to(device)
model = Transformer(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer_hi.token_to_id('[pad]'))
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    factor=0.1,
    patience=2,
    mode='min'
)

26000 26000


In [12]:
# Need to define scaler outside the loop first
scaler = torch.amp.GradScaler('cuda') 

def train(model, iterator, optimizer, criterion, clip, device):
    model.train()
    epoch_loss = 0
    
    loop = tqdm(enumerate(iterator), total=len(iterator), unit="batch")
    
    for i, (src, dec_input, trg_label) in loop:
        src = src.to(device)
        dec_input = dec_input.to(device)
        trg_label = trg_label.to(device)

        optimizer.zero_grad()
        
        # --- AMP Forward Pass (Saves Memory) ---
        with torch.amp.autocast('cuda'):
            output = model(src, dec_input)
            output_dim = output.shape[-1]
            
            output = output.reshape(-1, output_dim)
            trg_label = trg_label.reshape(-1)
            
            loss = criterion(output, trg_label)
        
        # --- AMP Backward Pass ---
        scaler.scale(loss).backward()
        
        # Unscale before clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        loop.set_description(f"Train Loss: {loss.item():.3f}")

    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion, device):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for i, (src, dec_input, trg_label) in enumerate(iterator):
            
            src = src.to(device)
            dec_input = dec_input.to(device)
            trg_label = trg_label.to(device)

            output = model(src, dec_input)

            output_dim = output.shape[-1]
            output = output.reshape(-1, output_dim)
            trg_label = trg_label.reshape(-1)

            loss = criterion(output, trg_label)
            epoch_loss += loss.item()
        
    return epoch_loss / len(iterator)

In [13]:
EPOCHS = 15
CLIP = 1  # Transformers can have exploding gradients, clipping helps

best_valid_loss = float('inf')
print(f"Starting training on {device}...")

for epoch in range(EPOCHS):
    
    start_time = time.time()
    
    # --- TRAIN ---
    train_loss = train(model, train_dataloader, optimizer, criterion, CLIP, device)
    
    # --- VALIDATE ---
    valid_loss = evaluate(model, val_dataloader, criterion, device)
    
    end_time = time.time()
    
    # Helper to format time
    epoch_mins = int((end_time - start_time) / 60)
    epoch_secs = int((end_time - start_time) % 60)
    
    # --- CHECKPOINTING ---
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'transformer_model.pt')
        save_msg = "--> Model Saved!"
    else:
        save_msg = ""
    
    # --- LOGGING ---
    # PPL (Perplexity) = exp(Loss). Lower is better.
    print(f'\nEpoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f} {save_msg}')
    
    # --- STEP SCHEDULER ---
    scheduler.step(valid_loss)
    
    # Print LR
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\tCurrent LR: {current_lr}")
    print("-" * 60)

Starting training on cuda...


Train Loss: 4.139: 100%|█████████████████| 6341/6341 [05:13<00:00, 20.21batch/s]



Epoch: 01 | Time: 5m 17s
	Train Loss: 4.890 | Train PPL: 132.930
	 Val. Loss: 4.229 |  Val. PPL:  68.623 --> Model Saved!
	Current LR: 0.0005
------------------------------------------------------------


Train Loss: 4.188: 100%|█████████████████| 6341/6341 [04:56<00:00, 21.37batch/s]



Epoch: 02 | Time: 4m 59s
	Train Loss: 4.063 | Train PPL:  58.175
	 Val. Loss: 3.895 |  Val. PPL:  49.168 --> Model Saved!
	Current LR: 0.0005
------------------------------------------------------------


...


Epoch: 11 | Time: 5m 0s
	Train Loss: nan | Train PPL:     nan
	 Val. Loss: 297.927 |  Val. PPL: 2444736661599045070199033725228893170068446401463699833681887489319397303150818455020026392714857904662228764649837511857777147904.000 
	Current LR: 0.0005
------------------------------------------------------------


Train Loss: 287.151: 100%|███████████████| 6341/6341 [05:04<00:00, 20.79batch/s]



Epoch: 12 | Time: 5m 9s
	Train Loss: nan | Train PPL:     nan
	 Val. Loss: 297.927 |  Val. PPL: 2444736661599045070199033725228893170068446401463699833681887489319397303150818455020026392714857904662228764649837511857777147904.000 
	Current LR: 5e-05
------------------------------------------------------------


Train Loss: 301.482:  14%|██▎             | 915/6341 [00:47<04:39, 19.40batch/s]


KeyboardInterrupt: 

In [14]:
import torch.nn.functional as F
state_dict = torch.load('transformer_model.pt')
model.load_state_dict(state_dict)

def predict_sentence(sentence, model, device, max_len=50, temperature=1.0, penalty=1.2):
    model.eval()
    
    # 1. Encode English
    encoded = tokenizer_en.encode(sentence)
    src = torch.LongTensor(encoded.ids).unsqueeze(0).to(device)
    src_mask = model.src_mask(src)
    
    # 2. Prepare Encoder Output (Compute once)
    with torch.no_grad():
        enc_src = model.encoder(src, src_mask)
    
    # 3. Initialize Decoder Input with [SOS]
    sos_id = tokenizer_hi.token_to_id("[sos]")
    eos_id = tokenizer_hi.token_to_id("[eos]")
    trg = torch.LongTensor([sos_id]).unsqueeze(0).to(device)
    
    generated_tokens = []
    
    for _ in range(max_len):
        with torch.no_grad():
            trg_mask = model.trg_mask(trg)
            
            # Get Decoder Output
            output = model.decoder(trg, enc_src, trg_mask, src_mask)
            
            # Get logits for the last token
            logits = output[:, -1, :] # Shape: [1, Vocab_Size]
            
            # --- APPLY REPETITION PENALTY ---
            # If a token has already been generated, lower its score
            for token in generated_tokens:
                # If score is positive, divide. If negative, multiply.
                if logits[0, token] < 0:
                    logits[0, token] *= penalty
                else:
                    logits[0, token] /= penalty
            
            # --- APPLY TEMPERATURE (Optional) ---
            # Higher temp (>1.0) = more random. Lower temp (<1.0) = more strict.
            logits = logits / temperature
            
            # Convert to probabilities
            probs = F.softmax(logits, dim=-1)
            
            # Pick the best token
            pred_token = torch.argmax(probs, dim=-1).item()
            
            # --- STOPPING ---
            if pred_token == eos_id:
                break
                
            # Append to list and inputs
            generated_tokens.append(pred_token)
            trg = torch.cat([trg, torch.LongTensor([[pred_token]]).to(device)], dim=1)

    return tokenizer_hi.decode(generated_tokens)

# Try running this
print(predict_sentence("my father is a very good man", model, device, penalty=1.5))

मेरे पिता है


In [15]:
test_sentences = [
    # --- 1. Basic Greeting & Introduction ---
    "Hello, how are you?",
    "My name is Rahul.",
    "Nice to meet you.",

    # --- 2. Simple Present & Continuous (Actions) ---
    "The cat is sitting on the mat.",
    "I am going to the market.",
    "She likes to read books.",
    "Children are playing in the park.",

    # --- 3. Questions (Who, What, Where, Why) ---
    "Where do you live?",
    "What are you doing?",
    "Why is he crying?",
    "When will you come home?",

    # --- 4. Negation (Not/Never) ---
    "I do not like tea.",
    "He did not go to school today.",
    "This is not a good idea.",

    # --- 5. Tenses (Past & Future) ---
    "I saw a movie yesterday.",
    "We will travel to Mumbai tomorrow.",
    "It rained heavily last night.",

    # --- 6. Complex/Longer Sentences (Conjunctions) ---
    "I cannot come to the party because I am sick.",
    "The food was tasty but very spicy.",
    "If it rains, we will stay inside.",
    
    # --- 7. Formal/Dataset Style (Likely in your 380k data) ---
    "The government is working on a new plan.",
    "This is a very serious matter.",
    "Education is important for everyone."
]

# Run the loop
print(f"{'ENGLISH':<50} | {'HINDI PREDICTION'}")
print("-" * 80)
for sent in test_sentences:
    # Use your new function with penalty=1.2 or 1.5
    translation = predict_sentence(sent, model, device, penalty=1.2)
    print(f"{sent:<50} | {translation}")

ENGLISH                                            | HINDI PREDICTION
--------------------------------------------------------------------------------
Hello, how are you?                                | कैसे हैं ?
My name is Rahul.                                  | मेरा नाम राहुल का नाम है ।
Nice to meet you.                                  | आप से मिलना चाहिए ।
The cat is sitting on the mat.                     | बिल्ली चूहे पर बैठ जाता है ।
I am going to the market.                          | मैं बाज़ार में मार्केट जा रहा हूं ।
She likes to read books.                           | वह किताबें पढ़ने की किताबों से पढ़ना पसंद है ।
Children are playing in the park.                  | बच्चों को पार्क में बच्चे पार्क में बाल बच रहे हैं ।
Where do you live?                                 | तुम कहाँ रहते हो ?
What are you doing?                                | आप क्या कर रहे हो ?
Why is he crying?                                  | वह चिल्ला रहे हैं ?
When will you come home?             

In [ ]:
print(predict_sentence("bazaar", model, device, penalty=1.5))

व्यापारिक भवन
